# Point 3: SQL & Data Quality
Connect the notebook to SQLite, load the required tables, and complete the exercises below.
For each query, add a brief comment explaining: (1) the business question it answers and (2) what data quality issue it could reveal.

In [22]:
!pip install pandas pyarrow ipython-sql


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# Conexión 


In [23]:
from pathlib import Path
import sqlite3

def create_connection(db_file):
    try:
        conn = sqlite3.connect(db_file)
        print(f"Connected to database: {db_file}")
        return conn
    except sqlite3.Error as e:
        print(e)
        return None

path = Path("data/processed/datalake.db")  # corrected spelling
path.parent.mkdir(parents=True, exist_ok=True)  # create directories if needed

conn = create_connection(path)

Connected to database: data/processed/datalake.db


In [24]:
import pandas as pd

# Point 2 result — BI output (joined, deduplicated, proximity-enriched)
bi_df = pd.read_parquet("../output/bi/")
bi_df.to_sql("bi_data", con=conn, if_exists="replace", index=False)
print(f"bi_data loaded: {len(bi_df):,} rows")

# Raw labels table (required by Point 3)
labels_df = pd.read_parquet("../data/raw/labels/")
labels_df.to_sql("labels_data", con=conn, if_exists="replace", index=False)
print(f"labels_data loaded: {len(labels_df):,} rows")

# Raw geo table (needed for coordinate-based quality checks)
geo_df = pd.read_parquet("../data/raw/geo/")
geo_df.to_sql("geo_data", con=conn, if_exists="replace", index=False)
print(f"geo_data loaded: {len(geo_df):,} rows")

conn.close()

bi_data loaded: 3,864 rows
labels_data loaded: 84,435 rows
geo_data loaded: 611,959 rows


In [25]:
# We will first load an sql extension into our environment
# This extension will allow us to work with sql on Colaboratory
#
%load_ext sql

# We will then connect to our in memory sqlite database
# NB: This database will cease to exist as soon as the database connection is closed.
# We will learn more about how databases are created later in prep.
#
%sql sqlite:///data/processed/datalake.db
# Nothing to do here

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [26]:
%%sql
-- Exercise 1a: NULL check on key fields of the result table (bi_data)
-- Business question: are there records with missing identity, location, or event classification?
-- Data quality issue: NULLs in these fields would silently exclude customers from analyses
--   or produce misleading aggregations. Reveals upstream ingestion or join failures.

SELECT
    COUNT(CASE WHEN customer_id IS NULL THEN 1 END) AS null_customer_id,
    COUNT(CASE WHEN comuna      IS NULL THEN 1 END) AS null_comuna,
    COUNT(CASE WHEN event_type  IS NULL THEN 1 END) AS null_event_type
FROM bi_data;

 * sqlite:///data/processed/datalake.db
Done.


null_customer_id,null_comuna,null_event_type
0,0,0


In [27]:
%%sql
-- Exercise 1b: Duplicate (customer_id, event_type) pairs after deduplication
-- Business question: did the ETL deduplication step produce a clean 1-row-per-customer-per-event-type output?
-- Data quality issue: duplicates would inflate nearby_count and distance aggregations,
--   distorting marketing segmentation and capacity planning reports.

SELECT customer_id, event_type, COUNT(*) AS occurrences
FROM bi_data
GROUP BY customer_id, event_type
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

 * sqlite:///data/processed/datalake.db
Done.


customer_id,event_type,occurrences


In [ ]:
%%sql
-- Exercise 1c: Coordinate outliers — customers outside their commune's bounding box
-- Business question: do any customers have GPS coordinates inconsistent with their declared commune?
-- Data quality issue: customers outside their commune's bbox likely have corrupt or swapped
--   coordinates, which would place them in the wrong spatial partition and produce incorrect
--   proximity results. 0 rows = clean data; any rows = records requiring investigation.

WITH commune_bbox AS (
    SELECT
        comuna,
        MIN(latitud) AS min_lat,
        MAX(latitud) AS max_lat,
        MIN(longitud) AS min_lon,
        MAX(longitud) AS max_lon
    FROM geo_data
    GROUP BY comuna
)

SELECT
    g.ID        AS customer_id,
    g.latitud,
    g.longitud,
    g.comuna,
    cb.min_lat, cb.max_lat,
    cb.min_lon, cb.max_lon
FROM geo_data g
JOIN commune_bbox cb ON g.comuna = cb.comuna
WHERE g.latitud  NOT BETWEEN cb.min_lat AND cb.max_lat
   OR g.longitud NOT BETWEEN cb.min_lon AND cb.max_lon
LIMIT 50;

 * sqlite:///data/processed/datalake.db
Done.


customer_id,latitud,longitud,comuna,min_lat,max_lat,min_lon,max_lon


In [29]:
%%sql
-- Exercise 2: Top 20 communes by type 2 events — window function ranking
-- Business question: which communes have the highest concentration of type 2 activity,
--   and what share of national volume do they represent? Informs network capacity planning
--   and prioritizes targeted marketing campaigns by geographic market size.
-- Data quality issue: communes with disproportionately large shares may indicate data
--   duplication or misattribution of events to a single commune code.

WITH type2_per_commune AS (
    SELECT
        g.comuna,
        COUNT(*) AS total_type2_events
    FROM labels_data l
    JOIN geo_data g ON l.ID = g.ID
    WHERE l.event = 2
    GROUP BY g.comuna
),
national_total AS (
    SELECT SUM(total_type2_events) AS total
    FROM type2_per_commune
)
SELECT
    t.comuna,
    t.total_type2_events,
    DENSE_RANK() OVER (ORDER BY t.total_type2_events DESC) AS national_rank,
    ROUND(100.0 * t.total_type2_events / n.total, 4)       AS pct_of_national_total
FROM type2_per_commune t
CROSS JOIN national_total n
ORDER BY national_rank
LIMIT 20;

 * sqlite:///data/processed/datalake.db
Done.


comuna,total_type2_events,national_rank,pct_of_national_total
8e7e23148e55a25a0a788a413727bcf5079c21bc5f7310187fd4132c15404052,983,1,6.309
4a8bc878fecae0db731883c790c3fdfc623220d388a990cf3be6429f214048ad,955,2,6.1293
630d8cf93464322637f0c27c04d65fd7565a04c43decc27f0f5685a689c23cb2,832,3,5.3398
b13b671cb296c1ce5eb94117f308118364cd258b322f61872cc7364dfcf5f2ad,804,4,5.1601
b8df8fb0f19ee92b800af6f7bd277b2c6c0f660ff2ea13ae5bdaba762f92abf2,804,4,5.1601
192579a17a7c6faf9fa70fcff6bc208aa9187dbe4ea70f69ffd991d180c1f823,690,5,4.4285
cc80409f27e2a1b5e3a0feeb3f976bdb13e93ce8144e686c8abdf6762ba6a1a7,642,6,4.1204
1663f043b1b1201a010d6965765c283b6e068ff0cd071fb7aa6ed1a7fd120172,596,7,3.8252
a532cdd0fac620d3932252b65681c993258383ce0ca78f4933e65185c3cfc462,428,8,2.7469
5029af19eafc8e0cf42b8bb6b2d05e7fb8c106e5926a2a03e541916e3f8812aa,389,9,2.4966


In [30]:
%%sql
-- Exercise 3: CTE — type 1 event geographic stats per commune, filter geographic outliers
-- Business question: which communes show statistically anomalous latitude distribution
--   for type 1 events? Communes deviating >10% from the national average may represent
--   geographic data errors, boundary misclassification, or genuine regional outliers
--   requiring separate treatment in segmentation models.
-- Data quality issue: extreme avg_lat deviation flags communes where event coordinates
--   may have been systematically shifted or where the commune code is shared across
--   geographically distant areas.

WITH type1_events AS (
    SELECT
        g.comuna,
        g.latitud,
        g.longitud
    FROM labels_data l
    JOIN geo_data g ON l.ID = g.ID
    WHERE l.event = 1
),
commune_stats AS (
    SELECT
        comuna,
        AVG(latitud)            AS avg_lat,
        MAX(latitud)            AS max_lat,
        MIN(latitud)            AS min_lat,
        AVG(longitud)           AS avg_lon,
        MAX(longitud)           AS max_lon,
        MIN(longitud)           AS min_lon,
        COUNT(*)                AS event_count,
        MAX(latitud) - MIN(latitud) AS lat_range
    FROM type1_events
    GROUP BY comuna
),
national_avg AS (
    SELECT AVG(avg_lat) AS national_avg_lat
    FROM commune_stats
)
SELECT
    cs.comuna,
    ROUND(cs.avg_lat,  2) AS avg_lat,
    ROUND(cs.max_lat,  2) AS max_lat,
    ROUND(cs.min_lat,  2) AS min_lat,
    ROUND(cs.avg_lon,  2) AS avg_lon,
    ROUND(cs.max_lon,  2) AS max_lon,
    ROUND(cs.min_lon,  2) AS min_lon,
    cs.event_count,
    ROUND(cs.lat_range, 2)                                               AS lat_range,
    ROUND(na.national_avg_lat, 2)                                        AS national_avg_lat,
    ROUND(ABS(cs.avg_lat - na.national_avg_lat) / ABS(na.national_avg_lat) * 100, 2) AS deviation_pct
FROM commune_stats cs
CROSS JOIN national_avg na
WHERE ABS(cs.avg_lat - na.national_avg_lat) / ABS(na.national_avg_lat) > 0.10
ORDER BY deviation_pct DESC;

 * sqlite:///data/processed/datalake.db
Done.


comuna,avg_lat,max_lat,min_lat,avg_lon,max_lon,min_lon,event_count,lat_range,national_avg_lat,deviation_pct
071f5915a5f86e38e578779469198ac892d1927cc9a426eac32c9d72dd9ffbb2,507101.52,510365.95,505156.56,7516576.95,7519004.77,7512947.14,108,5209.4,297553.71,70.42
34e5c00587885b9f15adff668497722b0d4b91d35e0071acc9bc170d40cf3598,131504.18,134657.1,130248.43,5911462.5,5916113.14,5907764.59,42,4408.66,297553.71,55.8
c51ed7a673a2184f2acf7c66d4bc0b82a55b9d8e8062b373749d428fc6389d90,134198.1,136576.95,130883.72,5923628.74,5928983.34,5919058.01,48,5693.23,297553.71,54.9
82688130a685a70bf81a96683837a4c693580241c8b1f4e818670ff51335f444,137633.02,140404.14,135836.16,5582411.11,5586891.64,5580082.98,21,4567.98,297553.71,53.75
87ed4f87a842f25728bbf37705ee61260473c042dee8c67308e3b7e9b5628028,139075.55,142517.69,136009.54,5918258.49,5922239.46,5914911.91,84,6508.15,297553.71,53.26
294852159550e2ecd610aa092e632b7048b90ec4f360fd9578d2cd72d617cdf8,141223.11,142598.7,139104.9,5906291.9,5913754.93,5903339.9,23,3493.8,297553.71,52.54
b0220271b4dc9684733c0380ea719a61556a6809f360b09a2bc5229dadcce2b1,143718.28,145210.28,141816.78,5927255.42,5929786.76,5923588.77,11,3393.5,297553.71,51.7
25417a5e239b6fd87c4f75f71149260f5faf315ed89eef2b04497f8811584248,149259.88,152467.34,146282.01,5499819.75,5501020.65,5497816.12,20,6185.33,297553.71,49.84
f51d43a75b06a6e6b3e891dc8462da0d5c8e1457075dfe589b58a84f10ee6440,166389.82,167850.99,165637.71,5417687.72,5418354.15,5416941.25,4,2213.28,297553.71,44.08
1aafe5df473e17de48e9327c08661869f0bcca639970d3297e274c49443ecfd9,169917.27,173194.48,165852.06,5402218.51,5404501.1,5399841.55,30,7342.42,297553.71,42.9


In [31]:
%%sql
-- Exercise 4: Conditional aggregation — type 1 vs type 2 events per commune, no JOINs
-- Business question: which communes have the highest type 2 / type 1 ratio?
--   A high ratio indicates areas with concentrated type 2 activity relative to type 1,
--   informing campaign targeting and antenna prioritization by event mix.
-- Data quality issue: NULL ratio (division by zero) reveals communes that received
--   type 2 events but have no type 1 events — possible classification error or
--   incomplete labeling pipeline.
-- Note: operates on bi_data (Point 2 result) which already carries event_type and
--   comuna — no JOIN required.

SELECT
    comuna,
    COUNT(CASE WHEN event_type = 1 THEN 1 END)                      AS type1_count,
    COUNT(CASE WHEN event_type = 2 THEN 1 END)                      AS type2_count,
    ROUND(
        1.0 * COUNT(CASE WHEN event_type = 2 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN event_type = 1 THEN 1 END), 0),
    4)                                                               AS type2_type1_ratio
FROM bi_data
GROUP BY comuna
ORDER BY type2_type1_ratio DESC NULLS LAST;

 * sqlite:///data/processed/datalake.db
Done.


comuna,type1_count,type2_count,type2_type1_ratio
1663f043b1b1201a010d6965765c283b6e068ff0cd071fb7aa6ed1a7fd120172,10,187,18.7
5029af19eafc8e0cf42b8bb6b2d05e7fb8c106e5926a2a03e541916e3f8812aa,7,99,14.1429
f3de540ebefb09aaf638e5adfe680d8470fb1e6cecca61512699b430bd53110c,1,11,11.0
0431501957fefa14ae758497b2d45c95854c570baca6a18f6b5071664b36240e,5,49,9.8
f2edf68ccfd45c7859c1a432a03e63dd831ba6b17bfcf3b7068612eed08830b8,3,27,9.0
5892b2be0e273c41d632bbc9005246db2a3463d030c32aceab5924f901a3ec48,5,45,9.0
0a03fdd5ab558d93c9ba86fc160697c07f2fdf3b187822ad603076d7175b5a16,1,9,9.0
8e7e23148e55a25a0a788a413727bcf5079c21bc5f7310187fd4132c15404052,22,192,8.7273
b13b671cb296c1ce5eb94117f308118364cd258b322f61872cc7364dfcf5f2ad,27,233,8.6296
afaa56a33178967316038a0b5e440828d11b78aac7faa772d6b5f6e7bb3017d0,2,17,8.5
